# figure3.ipynb

Manuscript analysis and plotting notebook.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
fdir1 = configured_path('reccap_mask_file')
mask = xr.open_mfdataset(fdir1)
mask_ocean = mask.open_ocean

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-1',
    end='2023-12-15',
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)
ds

In [ ]:
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap, ListedColormap
from matplotlib import font_manager
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.mpl.ticker as cticker
from pathlib import Path

# 注册项目内附带的真实 Times New Roman，并应用到所有图中文字。
font_dir = Path(configured_path('font_dir')).resolve()
font_files = [font_dir / 'Times New Roman.ttf', font_dir / 'Times New Roman Bold.ttf']
for font_file in font_files:
    if not font_file.is_file():
        raise FileNotFoundError(f'Required Times New Roman font not found: {font_file}')
    font_manager.fontManager.addfont(str(font_file))
resolved_times = Path(font_manager.findfont('Times New Roman', fallback_to_default=False)).resolve()
if resolved_times not in {font_file.resolve() for font_file in font_files}:
    raise RuntimeError(f'Times New Roman resolved to an unexpected file: {resolved_times}')
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'axes.unicode_minus': False
})

# Pred 和 Obs 使用相同海洋掩码以及相同时间、空间坐标。
mld_pred = ds['MLD_pred'].where(mask_ocean)
mld_obs = ds['MLD_true'].where(mask_ocean)
valid_pair = mld_pred.notnull() & mld_obs.notnull()

# (a, b) 分别为时间序列最大值；(c) 为最大值之差；(d) 为逐格 Pearson r。
pred_max = mld_pred.max('time', skipna=True).where(mld_pred.notnull().any('time'))
obs_max = mld_obs.max('time', skipna=True).where(mld_obs.notnull().any('time'))
max_difference = (pred_max - obs_max).where(valid_pair.any('time'))
pearson_r = xr.corr(mld_pred, mld_obs, dim='time').where(valid_pair.sum('time') >= 3)

def wrap_global_30e(da):
    """将经度重排到 [30, 390)，并补循环列消除 30°E/390°E 接缝。"""
    lon_wrapped = ((da['lon'] - 30) % 360) + 30
    wrapped = da.assign_coords(lon=lon_wrapped).sortby('lon')
    cyclic, cyclic_lon = add_cyclic_point(wrapped.values, coord=wrapped['lon'].values)
    return xr.DataArray(
        cyclic, coords={'lat': wrapped['lat'], 'lon': cyclic_lon},
        dims=('lat', 'lon'), name=da.name
    )

# (a, b) 换回 cmaps.precip_11lev[1:] 配色，并插值匹配全部分段。
mld_levels = np.array([10, 15, 25, 40, 50, 60, 70, 100, 120, 150, 200, 400, 600])
mld_cmap = cmaps.precip_11lev[1:].interp(13)
mld_norm = BoundaryNorm(mld_levels, ncolors=mld_cmap.N, extend='max')

# 最大值差固定为 -300--300 m，并插值为 20 个离散色块。
difference_levels = np.linspace(-300, 300, 21)
difference_cmap = LinearSegmentedColormap.from_list(
    'difference_no_white',
    ['#173b73', '#346aa0', '#68a6c3', '#b8d8dc', '#d9ead3',
     '#fee8b6', '#f4cf9d', '#ef9854', '#d94c36', '#971f2b'],
    N=20
)
difference_cmap.set_under('#173b73')
difference_cmap.set_over('#9e1f28')
difference_norm = BoundaryNorm(difference_levels, ncolors=difference_cmap.N)

# 相关系数范围为 0.4--1.0，共 12 个离散色块；低于 0.4 的值使用下限色。
correlation_levels = np.linspace(0.4, 1.0, 13)
correlation_cmap = cmaps.WhiteYellowOrangeRed[30:].interp(12)
correlation_norm = BoundaryNorm(correlation_levels, ncolors=correlation_cmap.N, clip=True)

projection = ccrs.Robinson(central_longitude=210)
data_crs = ccrs.PlateCarree()
fig, axes = plt.subplots(
    2, 2, figsize=(14.5, 10.5), dpi=150,
    subplot_kw={'projection': projection}, constrained_layout=False
)
fig.subplots_adjust(left=0.025, right=0.99, bottom=0.065, top=0.88,
                    wspace=0.055, hspace=0.28)

panels = [
    (pred_max, '(a) Maximum MLD (SJTU-ViT)', mld_cmap, mld_norm,
     mld_levels, 'max'),
    (obs_max, '(b) Maximum MLD (Obs)', mld_cmap, mld_norm,
     mld_levels, 'max'),
    (max_difference, '(c) Maximum MLD difference (SJTU-ViT − Obs)',
     difference_cmap, difference_norm, [-300, -150, 0, 150, 300], 'both'),
    (pearson_r, '(d) Pearson correlation coefficient (r)',
     correlation_cmap, correlation_norm, np.arange(0.4, 1.01, 0.1), 'neither')
]

for k, (ax, panel) in enumerate(zip(axes.flat, panels)):
    field, title, cmap, norm, ticks, extend = panel
    field_30e = wrap_global_30e(field)
    # d 图将 r < 0.5 的有效值压到最浅色；NaN 仍保持缺测。
    plot_field = field_30e.clip(min=correlation_levels[0]) if k == 3 else field_30e
    mesh = ax.contourf(
        plot_field['lon'], plot_field['lat'], plot_field.values,
        levels=norm.boundaries, transform=data_crs, cmap=cmap, norm=norm,
        extend=extend, antialiased=True, corner_mask=True
    )
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor='0.30', edgecolor='none', zorder=3)
    ax.coastlines(linewidth=0.45, color='0.25', zorder=4)
    ax.set_title(title, fontsize=20, pad=60)

    # 标签位置与 30°E--390°E 排列一致：90°E、150°W、30°W。
    label_sides = [['left', 'y'], False, ['left', 'y'], False][k]
    gl = ax.gridlines(
        crs=data_crs, draw_labels=label_sides, linewidth=0.35, color='0.5',
        alpha=0.45, linestyle='--', x_inline=False, y_inline=False
    )
    gl.xlocator = ticker.FixedLocator([90, 210, 330])
    gl.ylocator = ticker.FixedLocator([-80, -40, 0, 40, 80])
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 19, 'fontfamily': 'Times New Roman'}
    gl.ylabel_style = {'size': 19, 'fontfamily': 'Times New Roman'}
    if k >= 2:
        for x_pos, lon_label in zip([0.34, 0.50, 0.66], ['90°E', '150°W', '30°W']):
            ax.text(x_pos, -0.055, lon_label, transform=ax.transAxes, ha='center', va='top',
                    fontsize=19, fontfamily='Times New Roman')

    # 每个色条置于地图上方，与参考图布局一致。
    cax = inset_axes(ax, width=('70%' if k < 2 else '50%'), height='4.5%', loc='upper center',
                     bbox_to_anchor=(0, 0.105, 1, 1), bbox_transform=ax.transAxes,
                     borderpad=0)
    cbar = fig.colorbar(mesh, cax=cax, orientation='horizontal', ticks=ticks,
                        extend=extend, spacing='uniform')
    if k < 2:
        # 非线性色条按数据坐标手动固定上侧刻度，避免 inset 轴自动翻转。
        cbar.ax.set_xticks([])
        cbar.ax.xaxis.set_visible(False)
        tick_transform = cbar.ax.get_xaxis_transform()
        for tick_value in ticks:
            cbar.ax.plot([tick_value, tick_value], [1.0, 1.16], transform=tick_transform,
                         color='black', linewidth=0.8, clip_on=False)
            cbar.ax.text(tick_value, 1.22, f'{tick_value:g}', transform=tick_transform,
                         ha=('left' if tick_value == ticks[-1] else 'center'),
                         va='bottom', fontsize=16,
                         fontfamily='Times New Roman', clip_on=False)
    else:
        cbar.ax.xaxis.set_ticks_position('top')
        cbar.ax.tick_params(axis='x', top=True, labeltop=True, bottom=False, labelbottom=False,
                            labelsize=17, length=4, pad=2)
        for tick_label in cbar.ax.get_xticklabels():
            tick_label.set_fontfamily('Times New Roman')

output_file = Path(str(Path(configured_path('output_dir')) / 'figure3.png'))
output_file.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_file, dpi=600, bbox_inches='tight', pad_inches=0, facecolor='white')
plt.show()